# Big Data Analytics - JobStreet Indonesia
**Dataset:** 962 job listings IT dari JobStreet Indonesia
**Tools:** PySpark — DataFrame API, Spark SQL, MLlib, Structured Streaming

## 1. Setup & Inisialisasi Spark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, when, regexp_replace, regexp_extract, to_timestamp,
    count, avg, max, min, desc, round as spark_round
)
from pyspark.sql.types import DoubleType, IntegerType

spark = SparkSession.builder \
    .appName('JobStreetBigData') \
    .master('local[*]') \
    .config('spark.driver.memory', '2g') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')
print('Spark Version:', spark.version)
print('Spark UI    :', spark.sparkContext.uiWebUrl)

## 2. Load & Inspect Raw Data

In [ ]:
df_raw = spark.read.option('multiLine', True).json('jobstreet_jobs_final.json')
print('=== SCHEMA RAW ===')
df_raw.printSchema()
print(f'Total records: {df_raw.count()}')

## 3. Preprocessing & Cleaning

In [ ]:
# Flatten nested fields
df = df_raw.select(
    col('id'),
    col('title'),
    col('companyName'),
    col('advertiser.description').alias('advertiserName'),
    col('locations')[0]['label'].alias('location'),
    col('classifications')[0]['classification']['description'].alias('category'),
    col('classifications')[0]['subclassification']['description'].alias('subCategory'),
    col('salaryLabel'),
    col('workTypes')[0].alias('workType'),
    col('workArrangements.displayText').alias('workArrangement'),
    col('listingDate.dateTimeUtc').alias('postedAt'),
    col('isFeatured'),
    col('displayType'),
    col('teaser'),
)

# Cast timestamp
df = df.withColumn('postedAt', to_timestamp(col('postedAt')))

# Null handling
df = df.withColumn('salaryLabel',
    when((col('salaryLabel').isNull()) | (col('salaryLabel') == ''), 'Not Disclosed')
    .otherwise(col('salaryLabel')))

df = df.withColumn('workType',
    when(col('workType').isNull(), 'Unknown').otherwise(col('workType')))

# Ekstrak salary numerik:
# - Normalize: hapus \xa0 (non-breaking space), lalu strip separator (titik & koma)
df = df.withColumn('_salaryClean',
    when(col('salaryLabel') != 'Not Disclosed',
        regexp_replace(
            regexp_extract(
                regexp_replace(col('salaryLabel'), '\\u00a0', ' '),
                r'Rp\s?([\d.,]+)', 1
            ),
            '[.,]', ''
        )
    ).otherwise(None)
)
df = df.withColumn('salaryRaw',
    when((col('_salaryClean').isNotNull()) & (col('_salaryClean') != ''),
        col('_salaryClean').cast(DoubleType())
    ).otherwise(None)
).drop('_salaryClean')

# Salary period
df = df.withColumn('salaryPeriod',
    when(col('salaryLabel').contains('month'), 'per month')
    .when(col('salaryLabel').contains('year'), 'per year')
    .otherwise('Not Disclosed')
)

print('=== CLEANED SCHEMA ===')
df.printSchema()
print(f'Total records: {df.count()}')
df.show(5, truncate=False)

In [ ]:
# Cek missing values
print('=== MISSING VALUES PER KOLOM ===')
for c in df.columns:
    n = df.filter(col(c).isNull() | (col(c) == '')).count()
    pct = round(n / df.count() * 100, 2)
    print(f'  {c:25s}: {n:4d} ({pct}%)')

## 4. Analisis Batch — DataFrame API

In [ ]:
# 4.1 Top 10 perusahaan dengan lowongan terbanyak
print('=== TOP 10 PERUSAHAAN ===')
df.groupBy('companyName') \
  .agg(count('id').alias('total_jobs')) \
  .orderBy(desc('total_jobs')) \
  .show(10, truncate=False)

In [ ]:
# 4.2 Distribusi lokasi
print('=== TOP 15 LOKASI ===')
df.groupBy('location') \
  .agg(count('id').alias('total_jobs')) \
  .orderBy(desc('total_jobs')) \
  .show(15, truncate=False)

In [ ]:
# 4.3 Distribusi work type
print('=== DISTRIBUSI WORK TYPE ===')
total = df.count()
df.groupBy('workType') \
  .agg(count('id').alias('total')) \
  .withColumn('persentase_%', spark_round(col('total') / total * 100, 2)) \
  .orderBy(desc('total')) \
  .show(truncate=False)

In [ ]:
# 4.4 Top sub-kategori IT
print('=== TOP 10 SUB-KATEGORI ===')
df.groupBy('category', 'subCategory') \
  .agg(count('id').alias('total')) \
  .orderBy(desc('total')) \
  .show(10, truncate=False)

In [ ]:
# 4.5 Analisis salary
df_salary = df.filter(col('salaryRaw').isNotNull())
print(f'Jobs dengan info salary: {df_salary.count()} dari {df.count()}')

print('\n=== STATISTIK SALARY (Rp) ===')
df_salary.agg(
    spark_round(avg('salaryRaw'), 0).alias('rata_rata'),
    spark_round(min('salaryRaw'), 0).alias('minimum'),
    spark_round(max('salaryRaw'), 0).alias('maksimum'),
).show()

print('=== RATA-RATA SALARY PER SUB-KATEGORI ===')
df_salary.groupBy('subCategory') \
  .agg(count('id').alias('n_jobs'), spark_round(avg('salaryRaw'), 0).alias('avg_salary_min')) \
  .filter(col('n_jobs') >= 2) \
  .orderBy(desc('avg_salary_min')) \
  .show(15, truncate=False)

In [ ]:
# 4.6 Trend posting per tanggal
print('=== TREND POSTING PER TANGGAL (14 hari terakhir) ===')
df.withColumn('tanggal', F.to_date('postedAt')) \
  .groupBy('tanggal') \
  .agg(count('id').alias('jobs_posted')) \
  .orderBy(desc('tanggal')) \
  .show(14, truncate=False)

In [ ]:
# 4.7 Featured vs Non-Featured
print('=== FEATURED VS NON-FEATURED ===')
df.groupBy('isFeatured') \
  .agg(count('id').alias('total')) \
  .withColumn('persentase_%', spark_round(col('total') / df.count() * 100, 2)) \
  .show()

## 5. Spark SQL

In [ ]:
df.createOrReplaceTempView('jobs')
df_salary.createOrReplaceTempView('jobs_with_salary')
print('Temp views registered: jobs, jobs_with_salary')

In [ ]:
# SQL 1: Top 10 kota dengan rata-rata salary tertinggi
spark.sql("""
    SELECT location,
           COUNT(id)                  AS total_jobs,
           ROUND(AVG(salaryRaw), 0)   AS avg_salary_min
    FROM   jobs_with_salary
    GROUP  BY location
    HAVING COUNT(id) >= 2
    ORDER  BY avg_salary_min DESC
    LIMIT  10
""").show(truncate=False)

In [ ]:
# SQL 2: Distribusi workType per 5 kota terbesar
spark.sql("""
    SELECT location, workType, COUNT(id) AS total
    FROM   jobs
    WHERE  location IN (
               SELECT location FROM jobs
               GROUP BY location
               ORDER BY COUNT(id) DESC
               LIMIT 5
           )
    GROUP  BY location, workType
    ORDER  BY location, total DESC
""").show(30, truncate=False)

In [ ]:
# SQL 3: Perusahaan dengan rata-rata salary tertinggi
spark.sql("""
    SELECT companyName,
           COUNT(id)                  AS total_jobs,
           ROUND(AVG(salaryRaw), 0)   AS avg_salary,
           ROUND(MAX(salaryRaw), 0)   AS max_salary
    FROM   jobs_with_salary
    GROUP  BY companyName
    HAVING COUNT(id) >= 2
    ORDER  BY avg_salary DESC
    LIMIT  10
""").show(truncate=False)

## 6. MLlib — KMeans Clustering

In [ ]:
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.clustering import KMeans
from pyspark.ml import Pipeline

df_ml = df.select('id', 'subCategory', 'workType', 'location', 'isFeatured').dropna()
df_ml = df_ml.withColumn('isFeaturedInt', col('isFeatured').cast(IntegerType()))
print(f'Data untuk clustering: {df_ml.count()} rows')

indexer_sub  = StringIndexer(inputCol='subCategory', outputCol='subCategoryIdx', handleInvalid='keep')
indexer_work = StringIndexer(inputCol='workType',    outputCol='workTypeIdx',    handleInvalid='keep')
indexer_loc  = StringIndexer(inputCol='location',    outputCol='locationIdx',    handleInvalid='keep')

assembler = VectorAssembler(
    inputCols=['subCategoryIdx', 'workTypeIdx', 'locationIdx', 'isFeaturedInt'],
    outputCol='features'
)

kmeans   = KMeans(k=4, seed=42, featuresCol='features', predictionCol='cluster')
pipeline = Pipeline(stages=[indexer_sub, indexer_work, indexer_loc, assembler, kmeans])
model    = pipeline.fit(df_ml)

df_clustered = model.transform(df_ml)

print('\n=== DISTRIBUSI CLUSTER ===')
df_clustered.groupBy('cluster').count().orderBy('cluster').show()

print(f'WSSSE: {model.stages[-1].summary.trainingCost:.4f}')

In [ ]:
# Profil tiap cluster
print('=== PROFIL CLUSTER — TOP SUB-KATEGORI ===')
for i in range(4):
    print(f'\n--- Cluster {i} ---')
    df_clustered.filter(col('cluster') == i) \
        .groupBy('subCategory').count() \
        .orderBy(desc('count')) \
        .show(5, truncate=False)

## 7. Structured Streaming — Simulasi

In [ ]:
import os, shutil, time, threading

STREAM_INPUT = '/tmp/stream_input'
STREAM_CKPT  = '/tmp/stream_checkpoint'

for d in [STREAM_INPUT, STREAM_CKPT]:
    shutil.rmtree(d, ignore_errors=True)
    os.makedirs(d)

stream_schema = df.schema

def produce_stream():
    rows = df.limit(200).toJSON().collect()
    batch_size = 50
    for i in range(0, len(rows), batch_size):
        batch = rows[i:i+batch_size]
        path  = f'{STREAM_INPUT}/batch_{i//batch_size}.json'
        with open(path, 'w') as f:
            f.write('[' + ','.join(batch) + ']')
        print(f'[Producer] Batch {i//batch_size} ditulis ({len(batch)} rows)')
        time.sleep(3)

threading.Thread(target=produce_stream, daemon=True).start()
print('Producer streaming dimulai...')

In [ ]:
# Baca stream dan agregasi realtime: count jobs per lokasi
df_stream = spark.readStream \
    .schema(stream_schema) \
    .option('multiLine', True) \
    .json(STREAM_INPUT)

stream_agg = df_stream.groupBy('location', 'workType') \
    .agg(count('id').alias('job_count'))

query = stream_agg.writeStream \
    .outputMode('complete') \
    .format('console') \
    .option('truncate', False) \
    .option('numRows', 10) \
    .option('checkpointLocation', STREAM_CKPT) \
    .start()

query.awaitTermination(20)
query.stop()
print('Streaming selesai.')

## 8. Simpan Output

In [ ]:
# CSV — cleaned data
df.coalesce(1).write.mode('overwrite') \
  .option('header', True).csv('/tmp/jobstreet_cleaned')

# Parquet — format Big Data
df.write.mode('overwrite').parquet('/tmp/jobstreet_parquet')

# CSV — hasil clustering
df_clustered.select('id', 'subCategory', 'workType', 'location', 'cluster') \
    .coalesce(1).write.mode('overwrite') \
    .option('header', True).csv('/tmp/jobstreet_clusters')

print('Output tersimpan:')
print('  /tmp/jobstreet_cleaned  -> CSV')
print('  /tmp/jobstreet_parquet  -> Parquet')
print('  /tmp/jobstreet_clusters -> Clustering CSV')

spark.stop()
print('Done.')